# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', 'N/A')}: {getattr(metadata, 'description', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @ids
record_sets = list(dataset.metadata.record_sets)
print("Available record sets and their @ids:")
for rec in record_sets:
    print(f"@id: {rec['@id']} | Name: {rec.get('name', rec['@id'])}")

# For illustration, print fields for the first record set
if record_sets:
    rec = record_sets[0]
    print(f"\nFields for record set {rec['@id']}: ")
    fields = rec.get('fields', [])
    for field in fields:
        print(f"  Field @id: {field['@id']}, Name: {field.get('name', field['@id'])}, Data type: {field.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Define the list of record set @ids (update this as needed based on previous cell's output)
record_set_ids = [rec['@id'] for rec in list(dataset.metadata.record_sets)]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set: {record_set_id}")
    else:
        print(f"No records found for record set: {record_set_id}")

# For demonstration, select the first available record set with data
selected_record_set_id = None
for rid in record_set_ids:
    if rid in dataframes:
        selected_record_set_id = rid
        break

if selected_record_set_id is not None:
    print(f"\nColumns in {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    print(dataframes[selected_record_set_id].head())
else:
    print("No record sets with data available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Use the DataFrame and selected record set from previous step
# Try to select a reasonable numeric field by searching for float/integer columns
df = None
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Try to auto-detect a likely numeric field
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    if numeric_field is None:
        # Try conversion on some known fields
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if np.issubdtype(df[col].dtype, np.number):
                    numeric_field = col
                    break
            except Exception:
                continue

    if numeric_field is not None:
        print(f"Using numeric field '{numeric_field}' for filtering and normalization.")
        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 10
        # Filter records above threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by a non-numeric column (e.g., 'accession' or similar)
        group_field = None
        for col in df.columns:
            if col != numeric_field and not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped by '{group_field}', showing mean of '{numeric_field}':")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualizes the distribution of the numeric field and its normalized variant
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and df is not None and numeric_field is not None:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
        plt.title(f"Normalized Distribution of {numeric_field} (Filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel("Count")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded the FAIR^2 Croissant schema-based dataset:
  - **Title**: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
  - Explored metadata structure and fields using exclusively `@id` references for record sets and fields.
  - Loaded and analyzed records for a selected record set, filtering and normalizing a representative numeric field.
  - Visualized distributions to gain an understanding of the field's values and grouped data by a key attribute where possible.

For deeper domain analysis or ML modeling, further curation or domain-knowledge-based selection of fields/criteria is recommended. Use the `mlcroissant` library's advanced features to seamlessly access and document FAIR datasets.